In [1]:
import pandas as pd

In [2]:
flag = '17_22'

In [3]:
df = pd.read_csv(f'../Dados/Determinantes/clima/clima_agg_{flag}_season.csv',index_col = 'Unnamed: 0')

df.head()

,geocode,season,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year,dengue_pattern
0,2700102,Autumn,24.936441,71.559392,528.5750,92,88,12,88,2017,Episódico/Epidêmico
1,2700102,Spring,26.661764,59.088583,104.5703,90,41,1,41,2017,Episódico/Epidêmico
2,2700102,Summer,27.669137,60.282593,390.1183,90,42,11,42,2017,Episódico/Epidêmico
3,2700102,Winter,23.509128,67.164870,217.3068,92,86,2,86,2017,Episódico/Epidêmico
4,2700201,Autumn,24.229479,83.254857,1000.2414,92,92,43,92,2017,Episódico/Epidêmico


In [4]:
df.shape

(133680, 11)

In [5]:
def get_features_by_season(df, geo, pat): 
    df_end = pd.DataFrame()

    df_end['geocode'] = [geo]

    df_end['dengue_pattern'] = [pat]
    
    for season in ['Autumn', 'Spring', 'Summer', 'Winter']: 
    
        df_ = pd.DataFrame(df.loc[(df.geocode == geo) & (df.season == season)][['temp_med', 'umid_med', 'precip_tot', 'thr_temp',
           'thr_umid', 'thr_prec', 'thr_temp_umid']].mean()).T

        df_.columns = df_.columns + '_' + season

        df_end = pd.concat([df_end, df_], axis = 1)
        
    return df_end 

In [6]:
%%time
df_base = df[['geocode', 'dengue_pattern']].drop_duplicates().reset_index(drop = True)

df_final = pd.DataFrame()

for geo, pat in zip(df_base['geocode'], df_base['dengue_pattern']):

    df_final = pd.concat([df_final, 
                         get_features_by_season(df, geo=geo, pat=pat)],
                        ignore_index = True, axis =0)

    
df_final.head()

CPU times: user 2min 8s, sys: 1.12 s, total: 2min 9s
Wall time: 2min 25s


,geocode,dengue_pattern,temp_med_Autumn,umid_med_Autumn,precip_tot_Autumn,thr_temp_Autumn,thr_umid_Autumn,thr_prec_Autumn,thr_temp_umid_Autumn,temp_med_Spring,...,thr_umid_Summer,thr_prec_Summer,thr_temp_umid_Summer,temp_med_Winter,umid_med_Winter,precip_tot_Winter,thr_temp_Winter,thr_umid_Winter,thr_prec_Winter,thr_temp_umid_Winter
0,2700102,Episódico/Epidêmico,24.936707,74.109714,884.738917,92.0,87.833333,21.500000,87.833333,26.984375,...,46.000000,11.666667,46.000000,22.871859,72.904494,476.891583,92.0,89.333333,10.833333,89.333333
1,2700201,Episódico/Epidêmico,24.581218,83.391198,1517.581250,92.0,92.000000,43.666667,92.000000,25.203772,...,90.166667,15.333333,90.166667,22.510309,82.771253,1156.583100,92.0,92.000000,43.000000,92.000000
2,2700300,Epidêmico,24.732184,80.802415,1325.386833,92.0,91.833333,37.833333,91.833333,25.756471,...,89.166667,12.000000,89.166667,22.568477,80.694762,883.321133,92.0,92.000000,31.833333,92.000000
3,2700409,Episódico,24.770729,84.299913,1765.574050,92.0,92.000000,55.333333,92.000000,25.232037,...,90.166667,23.833333,90.166667,22.795154,83.148432,1377.794450,92.0,92.000000,53.666667,92.000000
4,2700508,Episódico,25.607514,82.767903,2567.648633,92.0,92.000000,64.666667,92.000000,25.651127,...,90.166667,39.166667,90.166667,24.023651,79.418239,1815.517683,92.0,92.000000,59.666667,92.000000


In [7]:
df_final.to_csv(f'../Dados/Determinantes/clima/clima_org_model_{flag}_season.csv')